In [1]:
import chess, chess.engine, os, stat
from policy import *
import random
from discrim import *
from file_helper import truncateGame

2026-04-22 12:05:43.366954: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-22 12:05:43.398259: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-22 12:05:45.537000: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/storage/icds/RISE/sw8/anaconda/conda_envs/pytorch/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A Nu

POLICY V7


In [2]:
from stockfish import Stockfish
engine_path = r"./stockfish/src/stockfish"
sf = Stockfish(engine_path, parameters={"Threads": 1, "Hash": 256})
sf.set_depth(2)
sf.set_skill_level(2)
sf.get_engine_parameters()

{'Debug Log File': '',
 'Threads': 1,
 'Hash': 256,
 'Ponder': False,
 'MultiPV': 1,
 'Skill Level': 2,
 'Move Overhead': 10,
 'Slow Mover': 100,
 'UCI_Chess960': False,
 'UCI_LimitStrength': False,
 'UCI_Elo': 1350,
 'Contempt': 0,
 'Min Split Depth': 0,
 'Minimum Thinking Time': 20}

In [3]:
games= load_json("./data/Bijay_1549_games.json")
print(len(games))

Loading games: 100%|██████████| 392/392 [00:00<00:00, 732.95it/s]

392


In [4]:
agent = Agent("Bijay_1549",stockfish_path=r"./stockfish/src/stockfish")
agent.train(games)

Agent V2
[]
Epoch 1/10
396/396 [==============================] - 177s 443ms/step - loss: 0.0027 - accuracy: 0.0835
Epoch 2/10
396/396 [==============================] - 177s 447ms/step - loss: 0.0019 - accuracy: 0.0896
Epoch 3/10
396/396 [==============================] - 213s 538ms/step - loss: 0.0017 - accuracy: 0.0917
Epoch 4/10
396/396 [==============================] - 254s 642ms/step - loss: 0.0014 - accuracy: 0.0907
Epoch 5/10
396/396 [==============================] - 260s 656ms/step - loss: 0.0011 - accuracy: 0.0914
Epoch 6/10
396/396 [==============================] - 251s 633ms/step - loss: 8.4986e-04 - accuracy: 0.0920
Epoch 7/10
396/396 [==============================] - 252s 638ms/step - loss: 6.1186e-04 - accuracy: 0.0932
Epoch 8/10
396/396 [==============================] - 257s 650ms/step - loss: 4.3646e-04 - accuracy: 0.0932
Epoch 9/10
396/396 [==============================] - 251s 635ms/step - loss: 3.0776e-04 - accuracy: 0.0938
Epoch 10/10
396/396 [===============

In [5]:
import os

def simulate_games(agent, sf, num_games=400, file_dir="./data", file_postfix="0"):
    os.makedirs(file_dir, exist_ok=True)  # ✅ fix: create dir before writing
    
    games_data = []
    i = 0
    while i < num_games:
        board = chess.Board()
        moves = []
        aborted = False
        
        while not board.is_game_over():
            try:
                if board.turn == chess.WHITE:
                    move = agent.act(board)
                else:
                    sf.set_fen_position(board.fen())
                    best = sf.get_best_move()
                    if best is None:
                        aborted = True
                        break
                    move = chess.Move.from_uci(best)
            except Exception as e:
                print(f"[Game {i+1}] Error: {e}")
                aborted = True
                break
            
            board.push(move)
            moves.append(move.uci())
        
        if aborted:
            continue  # ✅ fix: skip broken games instead of saving them
        
        game_data = {
            "event": "Agent vs Stockfish",
            "round": i + 1,
            "white": f"Mimic Agent of {agent.id}",
            "black": "Stockfish",
            "result": board.result(),
            "moves": " ".join(moves)
        }
        if truncateGame(game_data):
            games_data.append(game_data)
            i += 1

    file_postfix = str(file_postfix).replace(".", "_")
    file_path = f"{file_dir}/{agent.id}_agent_vs_stockfish_{file_postfix}.json"
    with open(file_path, "w") as f:
        json.dump(games_data, f, indent=4)
    print(f"[Saved] {file_path}")  # ✅ confirms write succeeded
    return file_path

In [6]:
def overall_similarity_pipeline(json_A, json_B, player_A, player_B):

    print(f"\n--- FULL PIPELINE: {player_A} vs {player_B} ---\n")

    # ============================================================
    # 🔧 FIX: normalize JSON INSIDE PIPELINE (list → string)
    # ============================================================
    def normalize_json(path):
        with open(path, "r") as f:
            games = json.load(f)

        for g in games:
            if isinstance(g.get("moves"), list):
                g["moves"] = " ".join(g["moves"])

        return games

    # Write temporary cleaned versions (no external preprocessing step)
    import tempfile

    def write_temp(games):
        tmp = tempfile.NamedTemporaryFile(delete=False, mode="w", suffix=".json")
        json.dump(games, tmp)
        tmp.close()
        return tmp.name

    clean_A = write_temp(normalize_json(json_A))
    clean_B = write_temp(normalize_json(json_B))

    # ============================================================
    # ORIGINAL PIPELINE (UNCHANGED LOGIC BELOW)
    # ============================================================
    b_A, m_A, l_A = load_json_game_sequences(clean_A, player_A, 1.0)
    b_B, m_B, l_B = load_json_game_sequences(clean_B, player_B, 0.0)

    min_games = min(len(l_A), len(l_B))
    if min_games == 0:
        print("Not enough usable games.")
        return None

    b_A, m_A, l_A = b_A[:min_games], m_A[:min_games], l_A[:min_games]
    b_B, m_B, l_B = b_B[:min_games], m_B[:min_games], l_B[:min_games]

    raw_boards = b_A + b_B
    raw_moves  = m_A + m_B
    raw_labels = l_A + l_B

    combined = list(zip(raw_boards, raw_moves, raw_labels))
    random.shuffle(combined)
    raw_boards, raw_moves, raw_labels = zip(*combined)

    all_boards = np.array(raw_boards)
    all_moves  = np.array(raw_moves)
    all_labels = np.array(raw_labels)

    all_moves_onehot = tf.one_hot(all_moves, NUM_MOVES)

    model = build_style_classifier()
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001, clipnorm=1.0),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    )

    model.fit(
        x={"board_seq": all_boards, "move_seq": all_moves_onehot},
        y=all_labels,
        batch_size=32,
        epochs=20,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=1
    )

    similarity = compute_overall_similarity(
        clean_A, clean_B, player_A, player_B, model
    )

    print(f"Overall playstyle similarity: {similarity:.2f}%")
    os.remove(clean_A)
    os.remove(clean_B)
    return similarity

In [7]:
def hyper_tuning(agent, sf, num_games= 400,file_dir="./data",player_file_dir="./data"):
    a_values = np.linspace(0, 1, 11)[::-1]

    best_a = None
    best_score = float("-inf")
    player_file_path = f"{player_file_dir}/{agent.id}_games.json"

    for a in a_values:
        try:
            agent.a = a

            agent_file_path = simulate_games(
                agent,
                sf,
                num_games,
                file_dir=file_dir,
                file_postfix=f"_a_{a:.2f}"
            )

            score = overall_similarity_pipeline(
                player_file_path,
                agent_file_path,
                f"{agent.id}",
                f"Mimic Agent of {agent.id}"
            )

            print(f"a={a:.2f}, score={score:.3f}")

            if score > best_score:
                best_score = score
                best_a = a

        finally:
            # 🔥 CRITICAL: prevent Colab crashes
            import gc
            tf.keras.backend.clear_session()
            gc.collect()

    print(f"\nBest a: {best_a:.2f} (score={best_score:.3f})")

In [8]:
hyper_tuning(agent,sf,num_games= 100,file_dir="./data100")

/storage/home/jmy5612/model V7/policy.py:142: UserWarning: Note that even though you've set Stockfish to play on a weaker elo or skill level, get_evaluation will still return full strength Stockfish's evaluation of the position.
  info = self.sf.get_evaluation()


[Saved] ./data100/Bijay_1549_agent_vs_stockfish__a_1_00.json

--- FULL PIPELINE: Bijay_1549 vs Mimic Agent of Bijay_1549 ---

Epoch 1/20
5/5 [==============================] - 16s 3s/step - loss: 1.8083 - accuracy: 0.5938 - val_loss: 1.7923 - val_accuracy: 0.7250
Epoch 2/20
5/5 [==============================] - 11s 2s/step - loss: 1.7798 - accuracy: 0.7688 - val_loss: 1.7628 - val_accuracy: 0.8000
Epoch 3/20
5/5 [==============================] - 10s 2s/step - loss: 1.7477 - accuracy: 0.7875 - val_loss: 1.7285 - val_accuracy: 0.7500
Epoch 4/20
5/5 [==============================] - 10s 2s/step - loss: 1.7044 - accuracy: 0.8125 - val_loss: 1.6770 - val_accuracy: 0.8500
Epoch 5/20
5/5 [==============================] - 11s 2s/step - loss: 1.6345 - accuracy: 0.8813 - val_loss: 1.5915 - val_accuracy: 0.9000
Epoch 6/20
5/5 [==============================] - 10s 2s/step - loss: 1.5283 - accuracy: 0.9062 - val_loss: 1.4580 - val_accuracy: 0.9000
Epoch 7/20
5/5 [==============================

2026-04-22 13:03:26.578005: W tensorflow/core/data/root_dataset.cc:286] Optimization loop failed: CANCELLED: Operation was cancelled


[Saved] ./data100/Bijay_1549_agent_vs_stockfish__a_0_90.json

--- FULL PIPELINE: Bijay_1549 vs Mimic Agent of Bijay_1549 ---

Epoch 1/20
3/3 [==============================] - 11s 2s/step - loss: 1.8148 - accuracy: 0.6716 - val_loss: 1.8028 - val_accuracy: 0.5882
Epoch 2/20
3/3 [==============================] - 5s 2s/step - loss: 1.8002 - accuracy: 0.6866 - val_loss: 1.7864 - val_accuracy: 0.5882
Epoch 3/20
3/3 [==============================] - 5s 1s/step - loss: 1.7817 - accuracy: 0.7164 - val_loss: 1.7700 - val_accuracy: 0.7059
Epoch 4/20
3/3 [==============================] - 5s 1s/step - loss: 1.7663 - accuracy: 0.7761 - val_loss: 1.7531 - val_accuracy: 0.7059
Epoch 5/20
3/3 [==============================] - 5s 1s/step - loss: 1.7486 - accuracy: 0.7313 - val_loss: 1.7349 - val_accuracy: 0.7647
Epoch 6/20
3/3 [==============================] - 5s 2s/step - loss: 1.7274 - accuracy: 0.6567 - val_loss: 1.7140 - val_accuracy: 0.8235
Epoch 7/20
3/3 [==============================] - 5